# 06 — scATAC-seq with ArchR

**Run on: x86** — Requires Cell Ranger ATAC output. Run `scripts/python/download_cellranger_data.py` on x86, then `scripts/python/run_cellranger.py` or `cellranger-atac count` -> fragments.tsv.gz.

**Target:** Figs 3I-O, 4A-N — Super-enhancers, Il1b peaks, accessibility

## Datasets

- Whole heart: TAC WT vs TAC Brd4KO
- CD45+ nuclei: Sham, TAC, TAC_Brd4KO

## Pipeline

1. Create ArchR project, add fragments
2. QC: minTSS 15-16, minFrags 1584-3163
3. addIterativeLSI -> Harmony -> clusters
4. Peak calling, DARs, super-enhancers

In [ ]:
# Setup (R)
suppressPackageStartupMessages({
  library(ArchR)
  library(ggplot2)
})

# Keep notebook output compact
options(progressr.enable = FALSE)

addArchRThreads(threads = max(1, parallel::detectCores() %/% 2))
addArchRGenome("mm10")

out_base <- file.path("..", "output", "cellranger")

# Minimal Fig 3/4-compatible subset (update as needed)
runs_atac_min <- c(
  Sham = "SRR22882163",
  TAC = "SRR22882164",
  TAC_Brd4KO = "SRR22882165"
)


In [ ]:
# Locate Cell Ranger ATAC fragments

find_fragments <- function(run_id, base_dir = out_base) {
  candidates <- c(
    file.path(base_dir, run_id, "outs", "fragments.tsv.gz"),
    file.path(base_dir, run_id, "outs", "atac_fragments.tsv.gz")
  )
  for (p in candidates) {
    if (file.exists(p)) return(p)
  }
  NA_character_
}

frag_paths <- vapply(unname(runs_atac_min), find_fragments, character(1))
frag_tbl <- data.frame(condition = names(runs_atac_min), run_id = unname(runs_atac_min), fragments = frag_paths)
print(frag_tbl)

if (all(is.na(frag_paths))) {
  stop(
    "No scATAC fragment files found under ", out_base, "\n\n",
    "Expected: output/cellranger/<SRR>/outs/fragments.tsv.gz\n",
    "Run (on x86): python scripts/python/run_cellranger.py --ref-atac data/refs/refdata-cellranger-arc-GRCm39-2024-A"
  )
}

frag_tbl <- frag_tbl[!is.na(frag_tbl$fragments), , drop = FALSE]


In [ ]:
# Build Arrow files + ArchR project

arrow_files <- createArrowFiles(
  inputFiles = setNames(frag_tbl$fragments, frag_tbl$run_id),
  sampleNames = frag_tbl$run_id,
  minTSS = 15,
  minFrags = 1500,
  addTileMat = TRUE,
  addGeneScoreMat = TRUE,
  force = FALSE
)

proj <- ArchRProject(
  ArrowFiles = arrow_files,
  outputDirectory = file.path("..", "output", "archr", "proj_min"),
  copyArrows = TRUE
)

proj$condition <- frag_tbl$condition[match(proj$Sample, frag_tbl$run_id)]
proj


In [ ]:
# QC visualization (TSS enrichment + fragments)

p_tss <- plotGroups(
  ArchRProj = proj,
  groupBy = "condition",
  colorBy = "cellColData",
  name = "TSSEnrichment"
)

p_frags <- plotGroups(
  ArchRProj = proj,
  groupBy = "condition",
  colorBy = "cellColData",
  name = "log10(nFrags)"
)

p_tss
p_frags


## Interpretation of scATAC QC plots

- Higher `TSSEnrichment` indicates cleaner chromatin accessibility signal around promoters, which generally reflects better nucleus quality.
- Higher `log10(nFrags)` reflects deeper per-cell coverage; very low-fragment cells are usually low-information and can destabilize downstream clustering.
- Compare condition distributions to check for strong technical imbalance before biological interpretation.
- Practical takeaway: proceed when all groups retain a broad high-quality core, while documenting any condition-specific QC skew.

In [ ]:
# Dimensionality reduction + clustering

proj <- addIterativeLSI(
  ArchRProj = proj,
  useMatrix = "TileMatrix",
  name = "IterativeLSI",
  iterations = 2,
  clusterParams = list(resolution = c(0.2), sampleCells = 5000, n.start = 10),
  varFeatures = 25000,
  dimsToUse = 1:30,
  force = TRUE
)

proj <- addHarmony(
  ArchRProj = proj,
  reducedDims = "IterativeLSI",
  name = "Harmony",
  groupBy = "condition",
  force = TRUE
)

proj <- addClusters(ArchRProj = proj, reducedDims = "Harmony", name = "Clusters", resolution = 0.6, force = TRUE)
proj <- addUMAP(ArchRProj = proj, reducedDims = "Harmony", name = "UMAP", nNeighbors = 30, minDist = 0.5, metric = "cosine", force = TRUE)

p_clusters <- plotEmbedding(ArchRProj = proj, colorBy = "cellColData", name = "Clusters", embedding = "UMAP")
p_condition <- plotEmbedding(ArchRProj = proj, colorBy = "cellColData", name = "condition", embedding = "UMAP")

p_clusters
p_condition


## Interpretation of UMAP and condition overlays

- Distinct UMAP islands suggest separable chromatin states/cell populations rather than a single continuum.
- Condition mixing across clusters suggests shared major cell identities across Sham/TAC/TAC_Brd4KO.
- Condition-enriched islands can indicate remodeling-associated states, but should be validated by differential accessibility and replicate-aware testing.
- Practical takeaway: use this map for annotation and hypothesis generation, then confirm with DAR/motif analyses.

In [ ]:
# Optional downstream hooks (DAR + motifs)

# proj <- addGroupCoverages(proj, groupBy = "condition")
# proj <- addReproduciblePeakSet(proj, groupBy = "condition", pathToMacs2 = "macs2")
# proj <- addPeakMatrix(proj)
# markersPeaks <- getMarkerFeatures(
#   ArchRProj = proj,
#   useMatrix = "PeakMatrix",
#   groupBy = "condition",
#   bias = c("TSSEnrichment", "log10(nFrags)"),
#   testMethod = "wilcoxon"
# )
# markersPeaks


## Practical tips (Cell Ranger ATAC + ArchR)

- Ensure each selected run has `output/cellranger/<SRR>/outs/fragments.tsv.gz`.
- Start with a minimal subset for figure prototyping, then scale to all runs.
- If runtime is slow, reduce temporary cell count in LSI (`sampleCells`) and increase later.
- Treat initial cluster/condition shifts as hypothesis-generating until confirmed with DAR/motif evidence and biological replication.